In [39]:
import os 
import pandas as pd
import multiprocessing
import requests
from sec_api import QueryApi, ExtractorApi

In [40]:
# Intitialize the API
sec_api_key = os.getenv('SEC_API_KEY')
query_api = QueryApi(sec_api_key)

number_of_processes = 2

### Get the 10K and 10Q URL form the API

In [29]:
# Define the tickers to analyze
tickers = ["IBIT", "ETHA", "FBTC", "FETH", "GBTC", "ETHE"]

# Set the ticker list to the correct format 
ticker_query = ", ".join(tickers)

# Query to the API
query = {
    "query": f'ticker:({ticker_query}) AND (formType:"10-K" OR formType:"10-Q")',
    "from": "0",
    "size": "50",
    "sort": [
        {
            "filedAt": {
                "order": "asc"
            }
        }
    ]
}

response = query_api.get_filings(query)

In [36]:
# Convert to a Dataframe
rows = []

for filing in response["filings"]:
    rows.append({
        "ticker": filing["ticker"],
        "company": filing["companyName"],
        "form": filing["formType"],
        "filed_at": filing["filedAt"],
        "period": filing["periodOfReport"],
        "url": filing["linkToFilingDetails"]
    })

df = pd.DataFrame(rows)

In [41]:
df.head()

,ticker,company,form,filed_at,period,url
0,FBTC,FIRST BANCTRUST CORP,10-Q,2005-05-13T14:39:16-04:00,2005-03-31,https://www.sec.gov/Archives/edgar/data/112984...
1,FBTC,FIRST BANCTRUST CORP,10-Q,2005-08-12T11:53:03-04:00,2005-06-30,https://www.sec.gov/Archives/edgar/data/112984...
2,FBTC,FIRST BANCTRUST CORP,10-Q,2005-11-10T17:11:45-05:00,2005-09-30,https://www.sec.gov/Archives/edgar/data/112984...
3,FBTC,FIRST BANCTRUST CORP,10-K,2006-03-28T13:15:40-05:00,2005-12-31,https://www.sec.gov/Archives/edgar/data/112984...
4,FBTC,FIRST BANCTRUST CORP,10-Q,2006-05-15T10:46:52-04:00,2006-03-31,https://www.sec.gov/Archives/edgar/data/112984...


### Extract the Data from the URLs